In [19]:
import os
from tensorflow.keras.preprocessing.text import text_to_word_sequence, Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [20]:
DATA_URL = "datasets/data.txt"
DATA_PATH = "data.txt"

In [21]:
def build_and_train_shakespeare_model(data_path=DATA_PATH, epochs=500):
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"{data_path} not found. "f"Download it from {DATA_URL} and place it in the same folder.")
    # Read the text file as separate lines of text
    with open(data_path, "r", encoding="utf-8") as file:
        text = file.read()
        lines = text.lower().split("\n")
    print(f"Loaded {len(lines)} lines from {data_path}")
    # Tokenize the entire text into words
    words = text_to_word_sequence(text)
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(words)  # fit on words list
    vocabulary_size = len(tokenizer.word_index) + 1
    print(f"Vocabulary size: {vocabulary_size}")
    # Convert each line into a sequence of integer tokens
    sequences = tokenizer.texts_to_sequences(lines)
    subsequences = []
    for sequence in sequences:
        # for a line of tokens [w1, w2, w3, w4]
        # we create: [w1, w2], [w1, w2, w3], [w1, w2, w3, w4]
        for i in range(1, len(sequence)):
            subsequence = sequence[: i + 1]
            subsequences.append(subsequence)
    print(f"Number of subsequences: {len(subsequences)}")
    if not subsequences:
        raise ValueError("No subsequences were created. Check the input text and tokenization.")
    sequence_length = max(len(seq) for seq in sequences)  # max original line length
    print(f"Max sequence length (original lines): {sequence_length}")
    # Pad subsequences so they all have the same length
    sequences_padded = pad_sequences(subsequences, maxlen=sequence_length, padding="pre")
    # x = all tokens except the last one
    # y = the last token (next word) -> one-hot
    x, y = sequences_padded[:, :-1], sequences_padded[:, -1]
    y = to_categorical(y, num_classes=vocabulary_size)
    print("Input shape:", x.shape)
    print("Target shape:", y.shape)
    model = Sequential()
    # Embedding layer
    model.add(Embedding(input_dim=vocabulary_size, output_dim=100, input_length=sequence_length - 1,))
    # LSTM layer
    model.add(LSTM(100))
    # Dropout layer (10%)
    model.add(Dropout(0.1))
    # Dense softmax output over vocabulary
    model.add(Dense(vocabulary_size, activation="softmax"))
    model.summary()
    model.compile(optimizer=Adam(), loss="categorical_crossentropy", metrics=["accuracy"],)
    print(f"Training for {epochs} epochs...")
    model.fit(x, y, epochs=epochs, verbose=1)
    # Return the trained model (as the exercise requires)
    return model, tokenizer, sequence_length

In [31]:
if __name__ == "__main__":
    model, tokenizer, seq_len = build_and_train_shakespeare_model("datasets/data.txt", epochs=10)  # use 10 first to test
    model.save("shakespeare_next_word_model.keras")
    print("Model saved to shakespeare_next_word_model.keras")

Loaded 2469 lines from datasets/data.txt
Vocabulary size: 3201
Number of subsequences: 15517
Max sequence length (original lines): 11
Input shape: (15517, 10)
Target shape: (15517, 3201)


c:\Users\mosta\anaconda3\envs\tf_env\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Training for 10 epochs...
Epoch 1/10
485/485 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.0249 - loss: 6.8446
Epoch 2/10
485/485 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.0291 - loss: 6.4633
Epoch 3/10
485/485 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.0342 - loss: 6.3190
Epoch 4/10
485/485 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.0421 - loss: 6.1543
Epoch 5/10
485/485 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.0491 - loss: 5.9923
Epoch 6/10
485/485 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.0570 - loss: 5.8264
Epoch 7/10
485/485 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.0683 - loss: 5.6501
Epoch 8/10
485/485 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.0745 - loss: 5.4736
Epoch 9/10
485/485 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.0838 - loss: 5.3001
Epoch 10/10
485/485 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.0918 - loss: 5.1285
Model saved to shakespeare_next_word_model.keras
